In [1]:
from pathlib import Path
import random

from IPython.display import display
import numpy as np
import pandas as pd
import torch


def find_project_root(start_path: Path) -> Path:
    for candidate in (start_path.resolve(), *start_path.resolve().parents):
        has_repo_markers = (candidate / ".github").exists() or (candidate / ".git").exists()
        if has_repo_markers and (candidate / "datasets").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate the repository root from notebook working directory: {start_path}"
    )


RANDOM_SEED = 42
PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "datasets" / "postprocessed-CHEMBL379_IC50" / "scaled_dataset"
TARGET_COLUMN = "activity"
METADATA_COLUMNS = [
    "representative_molecule_chembl_id",
    "smiles",
    "canonical_smiles",
    "activity",
    "threshold_nm",
    "source_record_count",
    "source_molecule_chembl_ids",
    "source_relations",
    "source_values_nm",
    "label_sources",
    "target_chembl_id",
]

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Expected scaled dataset directory was not found: {DATA_DIR}")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Scaled dataset directory: {DATA_DIR}")
print(f"Using device: {device}")

Project root: C:\Users\victo\VSCodeProjects\Magestic-NN
Scaled dataset directory: C:\Users\victo\VSCodeProjects\Magestic-NN\datasets\postprocessed-CHEMBL379_IC50\scaled_dataset
Using device: cuda


In [2]:
train_df = pd.read_csv(DATA_DIR / "train_dataset.csv")
validation_df = pd.read_csv(DATA_DIR / "validation_dataset.csv")
test_df = pd.read_csv(DATA_DIR / "test_dataset.csv")

print("Loaded splits:")
print(f"train: {train_df.shape}")
print(f"validation: {validation_df.shape}")
print(f"test: {test_df.shape}")

Loaded splits:
train: (1686, 82)
validation: (211, 82)
test: (211, 82)


In [3]:
descriptor_columns = [
    column_name
    for column_name in train_df.columns
    if column_name not in METADATA_COLUMNS
]
validation_descriptor_columns = [
    column_name
    for column_name in validation_df.columns
    if column_name not in METADATA_COLUMNS
]
test_descriptor_columns = [
    column_name
    for column_name in test_df.columns
    if column_name not in METADATA_COLUMNS
]

assert TARGET_COLUMN in train_df.columns
assert TARGET_COLUMN in validation_df.columns
assert TARGET_COLUMN in test_df.columns
assert descriptor_columns == validation_descriptor_columns == test_descriptor_columns, (
    "Train, validation, and test splits do not share the same descriptor columns."
)
assert not train_df[descriptor_columns].isnull().any().any()
assert not validation_df[descriptor_columns].isnull().any().any()
assert not test_df[descriptor_columns].isnull().any().any()
assert set(train_df[TARGET_COLUMN].unique()) <= {0, 1}
assert set(validation_df[TARGET_COLUMN].unique()) <= {0, 1}
assert set(test_df[TARGET_COLUMN].unique()) <= {0, 1}

sanity_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(validation_df), len(test_df)],
        "active": [
            int(train_df[TARGET_COLUMN].sum()),
            int(validation_df[TARGET_COLUMN].sum()),
            int(test_df[TARGET_COLUMN].sum()),
        ],
        "inactive": [
            int((train_df[TARGET_COLUMN] == 0).sum()),
            int((validation_df[TARGET_COLUMN] == 0).sum()),
            int((test_df[TARGET_COLUMN] == 0).sum()),
        ],
    }
)

print(f"Descriptor column count: {len(descriptor_columns)}")
display(sanity_summary)

Descriptor column count: 71


,split,rows,active,inactive
0,train,1686,768,918
1,validation,211,96,115
2,test,211,96,115


In [4]:
train_metadata = train_df[METADATA_COLUMNS].copy()
validation_metadata = validation_df[METADATA_COLUMNS].copy()
test_metadata = test_df[METADATA_COLUMNS].copy()

X_train_df = train_df[descriptor_columns].copy()
X_validation_df = validation_df[descriptor_columns].copy()
X_test_df = test_df[descriptor_columns].copy()

y_train_series = train_df[TARGET_COLUMN].astype(np.float32)
y_validation_series = validation_df[TARGET_COLUMN].astype(np.float32)
y_test_series = test_df[TARGET_COLUMN].astype(np.float32)

X_train = torch.tensor(X_train_df.to_numpy(dtype=np.float32), dtype=torch.float32)
X_validation = torch.tensor(
    X_validation_df.to_numpy(dtype=np.float32),
    dtype=torch.float32,
 )
X_test = torch.tensor(X_test_df.to_numpy(dtype=np.float32), dtype=torch.float32)

y_train = torch.tensor(
    y_train_series.to_numpy(dtype=np.float32),
    dtype=torch.float32,
).unsqueeze(1)
y_validation = torch.tensor(
    y_validation_series.to_numpy(dtype=np.float32),
    dtype=torch.float32,
).unsqueeze(1)
y_test = torch.tensor(
    y_test_series.to_numpy(dtype=np.float32),
    dtype=torch.float32,
).unsqueeze(1)

print(f"X_train shape: {tuple(X_train.shape)}, y_train shape: {tuple(y_train.shape)}")
print(
    f"X_validation shape: {tuple(X_validation.shape)}, "
    f"y_validation shape: {tuple(y_validation.shape)}"
 )
print(f"X_test shape: {tuple(X_test.shape)}, y_test shape: {tuple(y_test.shape)}")

X_train shape: (1686, 71), y_train shape: (1686, 1)
X_validation shape: (211, 71), y_validation shape: (211, 1)
X_test shape: (211, 71), y_test shape: (211, 1)


In [5]:
from torch.utils.data import DataLoader, TensorDataset

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64

train_dataset = TensorDataset(X_train, y_train)
validation_dataset = TensorDataset(X_validation, y_validation)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)

sample_features, sample_targets = next(iter(train_loader))

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(validation_loader)}")
print(f"Test batches: {len(test_loader)}")
print(
    f"Sample training batch shapes: X={tuple(sample_features.shape)}, "
    f"y={tuple(sample_targets.shape)}"
 )

Train batches: 53
Validation batches: 4
Test batches: 4
Sample training batch shapes: X=(32, 71), y=(32, 1)


In [6]:
import torch.nn as nn

ARCHITECTURE_CANDIDATES = {
    "small_mlp": [32],
    "recommended_mlp": [64, 32],
    "large_mlp": [128, 64, 32],
}
DROPOUT_RATE = 0.2


class DescriptorMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float = 0.2):
        super().__init__()

        layers: list[nn.Module] = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.network(inputs)


def count_trainable_parameters(model: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


INPUT_DIM = X_train.shape[1]
architecture_summary = pd.DataFrame(
    [
        {
            "model": model_name,
            "hidden_layers": hidden_dims,
            "dropout": DROPOUT_RATE,
            "trainable_parameters": count_trainable_parameters(
                DescriptorMLP(INPUT_DIM, hidden_dims, dropout=DROPOUT_RATE)
            ),
        }
        for model_name, hidden_dims in ARCHITECTURE_CANDIDATES.items()
    ]
)

recommended_model = DescriptorMLP(
    INPUT_DIM,
    ARCHITECTURE_CANDIDATES["recommended_mlp"],
    dropout=DROPOUT_RATE,
).to(device)

display(architecture_summary)
print(recommended_model)

,model,hidden_layers,dropout,trainable_parameters
0,small_mlp,[32],0.2,2337
1,recommended_mlp,"[64, 32]",0.2,6721
2,large_mlp,"[128, 64, 32]",0.2,19585


DescriptorMLP(
  (network): Sequential(
    (0): Linear(in_features=71, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [7]:
from copy import deepcopy

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


def reset_random_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def collect_predictions(model: nn.Module, data_loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    collected_targets = []
    collected_logits = []

    with torch.no_grad():
        for batch_features, batch_targets in data_loader:
            batch_features = batch_features.to(device)
            batch_targets = batch_targets.to(device)
            batch_logits = model(batch_features)

            collected_targets.append(batch_targets.detach().cpu().numpy())
            collected_logits.append(batch_logits.detach().cpu().numpy())

    targets = np.vstack(collected_targets).reshape(-1)
    logits = np.vstack(collected_logits).reshape(-1)
    return targets, logits


def compute_binary_classification_metrics(
    targets: np.ndarray,
    logits: np.ndarray,
 ) -> dict[str, float]:
    probabilities = 1.0 / (1.0 + np.exp(-logits))
    predictions = (probabilities >= 0.5).astype(np.int64)

    return {
        "accuracy": float(accuracy_score(targets, predictions)),
        "precision": float(precision_score(targets, predictions, zero_division=0)),
        "recall": float(recall_score(targets, predictions, zero_division=0)),
        "f1": float(f1_score(targets, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(targets, probabilities)),
    }


def evaluate_model(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: nn.Module,
 ) -> dict[str, float]:
    model.eval()
    running_loss = 0.0
    total_examples = 0

    with torch.no_grad():
        for batch_features, batch_targets in data_loader:
            batch_features = batch_features.to(device)
            batch_targets = batch_targets.to(device)
            batch_logits = model(batch_features)
            batch_loss = loss_function(batch_logits, batch_targets)

            batch_size = batch_features.size(0)
            running_loss += batch_loss.item() * batch_size
            total_examples += batch_size

    targets, logits = collect_predictions(model, data_loader)
    metrics = compute_binary_classification_metrics(targets, logits)
    metrics["loss"] = running_loss / total_examples
    return metrics


def train_model(
    model: nn.Module,
    train_data_loader: DataLoader,
    validation_data_loader: DataLoader,
    learning_rate: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_PATIENCE,
 ) -> tuple[nn.Module, pd.DataFrame, dict[str, float]]:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    loss_function = nn.BCEWithLogitsLoss()

    best_model_state = deepcopy(model.state_dict())
    best_validation_metrics: dict[str, float] | None = None
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history_rows: list[dict[str, float | int]] = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_training_loss = 0.0
        total_training_examples = 0

        for batch_features, batch_targets in train_data_loader:
            batch_features = batch_features.to(device)
            batch_targets = batch_targets.to(device)

            optimizer.zero_grad()
            batch_logits = model(batch_features)
            batch_loss = loss_function(batch_logits, batch_targets)
            batch_loss.backward()
            optimizer.step()

            batch_size = batch_features.size(0)
            running_training_loss += batch_loss.item() * batch_size
            total_training_examples += batch_size

        train_loss = running_training_loss / total_training_examples
        validation_metrics = evaluate_model(
            model,
            validation_data_loader,
            loss_function,
        )

        history_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "validation_loss": validation_metrics["loss"],
                "validation_accuracy": validation_metrics["accuracy"],
                "validation_precision": validation_metrics["precision"],
                "validation_recall": validation_metrics["recall"],
                "validation_f1": validation_metrics["f1"],
                "validation_roc_auc": validation_metrics["roc_auc"],
            }
        )

        if validation_metrics["loss"] < best_validation_loss:
            best_validation_loss = validation_metrics["loss"]
            best_validation_metrics = validation_metrics.copy()
            best_model_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_model_state)
    history_frame = pd.DataFrame(history_rows)
    if best_validation_metrics is None:
        raise RuntimeError("Training finished without recording validation metrics.")

    return model, history_frame, best_validation_metrics


def run_experiment(
    experiment_name: str,
    hidden_dims: list[int],
    dropout: float = DROPOUT_RATE,
    learning_rate: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_PATIENCE,
 ) -> dict[str, object]:
    reset_random_seed(RANDOM_SEED)
    model = DescriptorMLP(INPUT_DIM, hidden_dims, dropout=dropout).to(device)
    trained_model, history_frame, best_validation_metrics = train_model(
        model,
        train_loader,
        validation_loader,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        patience=patience,
    )
    loss_function = nn.BCEWithLogitsLoss()
    test_metrics = evaluate_model(trained_model, test_loader, loss_function)

    summary = pd.DataFrame(
        [
            {
                "experiment": experiment_name,
                "hidden_layers": hidden_dims,
                "dropout": dropout,
                "epochs_trained": int(history_frame["epoch"].iloc[-1]),
                "validation_loss": best_validation_metrics["loss"],
                "validation_accuracy": best_validation_metrics["accuracy"],
                "validation_precision": best_validation_metrics["precision"],
                "validation_recall": best_validation_metrics["recall"],
                "validation_f1": best_validation_metrics["f1"],
                "validation_roc_auc": best_validation_metrics["roc_auc"],
                "test_loss": test_metrics["loss"],
                "test_accuracy": test_metrics["accuracy"],
                "test_precision": test_metrics["precision"],
                "test_recall": test_metrics["recall"],
                "test_f1": test_metrics["f1"],
                "test_roc_auc": test_metrics["roc_auc"],
            }
        ]
    )

    return {
        "model": trained_model,
        "history": history_frame,
        "summary": summary,
    }


print(
    "Training utilities ready. "
    f"Defaults: epochs={MAX_EPOCHS}, patience={EARLY_STOPPING_PATIENCE}, "
    f"lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}"
 )

Training utilities ready. Defaults: epochs=100, patience=15, lr=0.001, weight_decay=0.0001


In [8]:
small_experiment = run_experiment(
    "small_mlp",
    ARCHITECTURE_CANDIDATES["small_mlp"],
)

display(small_experiment["summary"])
display(small_experiment["history"].tail())

,experiment,hidden_layers,dropout,epochs_trained,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,test_loss,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,small_mlp,[32],0.2,44,0.375391,0.838863,0.816327,0.833333,0.824742,0.916938,0.319052,0.862559,0.819048,0.895833,0.855721,0.938723


,epoch,train_loss,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc
39,40,0.279911,0.391525,0.829384,0.812500,0.812500,0.812500,0.912500
40,41,0.293380,0.392228,0.843602,0.824742,0.833333,0.829016,0.912953
41,42,0.293579,0.392341,0.834123,0.827957,0.802083,0.814815,0.914040
42,43,0.282508,0.394707,0.834123,0.821053,0.812500,0.816754,0.913225
43,44,0.280570,0.396075,0.824645,0.804124,0.812500,0.808290,0.911957


In [9]:
recommended_experiment = run_experiment(
    "recommended_mlp",
    ARCHITECTURE_CANDIDATES["recommended_mlp"],
)

display(recommended_experiment["summary"])
display(recommended_experiment["history"].tail())

,experiment,hidden_layers,dropout,epochs_trained,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,test_loss,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,recommended_mlp,"[64, 32]",0.2,37,0.363605,0.862559,0.825243,0.885417,0.854271,0.927446,0.335986,0.872038,0.82243,0.916667,0.866995,0.930027


,epoch,train_loss,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc
32,33,0.242248,0.405895,0.862559,0.831683,0.875000,0.852792,0.917935
33,34,0.241067,0.419816,0.829384,0.826087,0.791667,0.808511,0.917210
34,35,0.244612,0.415127,0.838863,0.816327,0.833333,0.824742,0.916667
35,36,0.246669,0.412429,0.853081,0.828283,0.854167,0.841026,0.919384
36,37,0.236495,0.408565,0.838863,0.822917,0.822917,0.822917,0.920562


In [10]:
large_experiment = run_experiment(
    "large_mlp",
    ARCHITECTURE_CANDIDATES["large_mlp"],
)

display(large_experiment["summary"])
display(large_experiment["history"].tail())

,experiment,hidden_layers,dropout,epochs_trained,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,test_loss,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,large_mlp,"[128, 64, 32]",0.2,41,0.355698,0.85782,0.851064,0.833333,0.842105,0.933605,0.379687,0.838863,0.792453,0.875,0.831683,0.925861


,epoch,train_loss,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc
36,37,0.186419,0.491287,0.838863,0.860465,0.770833,0.813187,0.917210
37,38,0.175270,0.518967,0.834123,0.858824,0.760417,0.806630,0.909783
38,39,0.184905,0.510277,0.810427,0.791667,0.791667,0.791667,0.901993
39,40,0.175013,0.479783,0.834123,0.850575,0.770833,0.808743,0.921105
40,41,0.177994,0.515449,0.805687,0.795699,0.770833,0.783069,0.899728


In [11]:
from datetime import datetime
import json

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

experiment_names = [
    "small_experiment",
    "recommended_experiment",
    "large_experiment",
]

available_experiments = [
    globals()[name]
    for name in experiment_names
    if name in globals()
]

results_summary = pd.concat(
    [experiment["summary"] for experiment in available_experiments],
    ignore_index=True,
)

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

summary_path = RESULTS_DIR / f"deep_learning_results_{run_timestamp}.csv"
json_path = RESULTS_DIR / f"deep_learning_results_{run_timestamp}.json"

results_summary.to_csv(summary_path, index=False)

payload = {
    "run_timestamp": run_timestamp,
    "device": str(device),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "results": results_summary.to_dict(orient="records"),
}

with json_path.open("w", encoding="utf-8") as file:
    json.dump(payload, file, indent=2)

display(results_summary)
print(f"Saved CSV: {summary_path}")
print(f"Saved JSON: {json_path}")


,experiment,hidden_layers,dropout,epochs_trained,validation_loss,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,test_loss,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc
0,small_mlp,[32],0.2,44,0.375391,0.838863,0.816327,0.833333,0.824742,0.916938,0.319052,0.862559,0.819048,0.895833,0.855721,0.938723
1,recommended_mlp,"[64, 32]",0.2,37,0.363605,0.862559,0.825243,0.885417,0.854271,0.927446,0.335986,0.872038,0.822430,0.916667,0.866995,0.930027
2,large_mlp,"[128, 64, 32]",0.2,41,0.355698,0.857820,0.851064,0.833333,0.842105,0.933605,0.379687,0.838863,0.792453,0.875000,0.831683,0.925861


Saved CSV: C:\Users\victo\VSCodeProjects\Magestic-NN\results\deep_learning_results_2026-05-02_01-56-27.csv
Saved JSON: C:\Users\victo\VSCodeProjects\Magestic-NN\results\deep_learning_results_2026-05-02_01-56-27.json
